### Performing parameter studies with PINNs in TorchPhysics
We now consider the Stokes equation and want to learn the flow field for different injection angles $\alpha$. The equation is given by:

\begin{align*}
    -\frac{\mu}{2} \text{div}(\nabla u_\alpha + \nabla u_\alpha^T) + \nabla p_\alpha &= 0 &&\text{ in } [0, 1]^2  \quad \text{(Conservation of momentum)}, \\
    \text{div}(u_\alpha) &= 0 &&\text{ in } [0, 1]^2 \quad  \text{(Conservation of mass)}, \\
    u_\alpha &= u_{in}(\alpha, x) , &&\text{ on } \{0 \} \times [0, 1] \quad \text{(Inflow condition)},\\
    u_\alpha &= 0 , &&\text{ on } [0, 1] \times \{0, 1\} \quad \text{(Noslip condition)}, \\
    -\frac{\mu}{2}(\nabla u_\alpha + \nabla u_\alpha^T) \cdot n + p_\alpha n &= 0 , &&\text{ on }  \{1 \} \times [0, 1] \quad \text{(Outflow condition)}.
\end{align*}

This equation describes a channel, where at the left side we prescribe the velocity field, at the bottom and top have a noslip condition (where the fluid sticks to the wall) and at the right side a free outflow. You dont need to understand what each equation means in detail, some further information will be given below.

Our main goal is to learn, with one single neural network, multiple solutions for different $\alpha$. 

The inflow function will be given by
\begin{equation*}
    u_{in}(\alpha, x) = 4.0 x_2 (1.0 - x_2) \left(\begin{matrix}\cos(\alpha) \\ \sin(\alpha)\end{matrix} \right),
\end{equation*}
such that the flow direction is controlled by the angle and the velocity profile reaches 0 when we approach the bottom and top boundary. 

In [1]:
# This block is for GPU selection. Please execute.
# import pathlib
# import os
# user = int(str(pathlib.Path().resolve())[22:24])
# os.environ["CUDA_VISIBLE_DEVICES"]= str(user % 4)

In [19]:
import qewton
import math

path = "/localdata/tomfre/SolutionData/StokesData/"

# Here all parameters are defined:
h, w = 1, 1
mu = 0.5
speed_scale = 4.0

# Number of training points 
N_pde = 50000
N_in = 40000
N_out = 10000

First we need define the spaces of our problem. Note, we now need a space "X" for the domain, "A" for the inflow angles, "u" for the velocity output and "p" for the pressure!

Next ot the domain $\Omega$, we also need to sample different angles that will be used as an input to our neural network and in the inflow function.
For now, create the interval $[\pi/4.0, \pi/2.0]$ for our angles.

In [3]:
# TODO: Add the space for pressure and velocity
X = qewton.Variable('x', dim=2)
A = qewton.Variable("alpha", dim=1)
P = qewton.Variable('p', dim=1)
U = qewton.Variable('u', dim=2)

Omega = qewton.geometries.Rectangle(X, [0, 0], w, h)
# TODO: Create interval for the angles (use math.pi, for pi):
A_int = qewton.geometries.Interval(A, math.pi/4.0, math.pi/2.0)

Next are the samplers, which are almost complete. We again need to filter the points at the boundary, since we have multiple boundary conditions.
Here, we incoperate the Dirichlet conditions into one PINN-condition and therefore need only two samplers.

Finish the sampler that returns true if the point $x$ is on the right boundary, meaning $x_1 = 1$.

In [4]:
# TODO: Complete the filter function:
def right_boundary_filter(x : X):
    return x[:, :1] == w

def other_boundary_filter(x : X):
    return x[:, :1] != w

inner_sampler = qewton.RandomUniformSampler(Omega*A_int, N_pde)
dirichlet_boundary_sampler = qewton.RandomUniformSampler(A_int*Omega.boundary, 
                                                         N_in, 
                                                         filter_fn=other_boundary_filter)
outflow_boundary_sampler = qewton.RandomUniformSampler(A_int*Omega.boundary, 
                                                       N_out, 
                                                       filter_fn=right_boundary_filter)

Now, we can start with implementing all conditions. Lets start with the neural network, that should learn the solution consisting of $(u, p)$ for different $(x, \alpha)$.

In [5]:
# TODO: Add the correct input and output spaces
model = qewton.FCN(
    in_neurons=X*A,
    hidden_neurons=30,
    out_neurons=U*P,
    n_hidden_layers=4,
    activation=qewton.bb.Tanh,
)

The first condition, will be the conservation of mass meaning $\text{div}(u)=0$:

In [6]:
# TODO: Implement the residual and complete the PINNCondition.
# Use .div to compute the divergence of a vector field
def mass_residual(u: U, x: X): 
    return u.div(x)

mass_pipeline = qewton.PINNPipeline(inner_sampler, [model], 
                                 residual=mass_residual, 
                                 residual_name="MassConstraint", 
                                 weight=20.0)
mass_constraint = mass_pipeline.constraint

The second condition, is the conservation of momentum meaning $-\frac{\mu}{2} \text{div}(\nabla u + \nabla u^T) + \nabla p=0$:


In [7]:
def momentum_residual(u: U, p: P, x: X):  # type: ignore
    return mu * u.sym_grad(x).matrix_div(x) - p.gradient(x)

momentum_pipeline = qewton.PINNPipeline(inner_sampler, [model], 
                                     residual=momentum_residual, 
                                     residual_name="MomentumConstraint")
momentum_constraint = momentum_pipeline.constraint

Now, we need to handle the boundary conditions. Let start with the Dirichlet condition.

In the following cell, we prepared the boundary function. Note, that this function combines the inflow profile and the no slip condition into one. This simplifies the implementation of the boundary condition. The function returns the inflow profile if the point is on the left boundary and returns 0 if the point is on the top or bottom boundary.

In [8]:
zeros_like_node = qewton.bb.ZerosLike()
cos = qewton.bb.Cos()
sin = qewton.bb.Sin()

def inflow_fn(x, alpha):
    inflow_fn = zeros_like_node(x)
    poly_scaling = speed_scale * x[:, 1:] * (h - x[:, 1:])
    inflow_fn[:, :1] = cos(alpha) * poly_scaling
    inflow_fn[:, 1:] = sin(alpha) * poly_scaling
    return inflow_fn

In [9]:
# TODO: finish the residual for the Dirichlet condition.
def dirichlet_residual(u: U, x: X, alpha: A):
    inflow = inflow_fn(x, alpha)
    return u - inflow

# TODO: Use a PINNPipeline with a weight of 50.0!
dirichlet_pipeline = qewton.PINNPipeline(dirichlet_boundary_sampler, 
                                     models=[model], 
                                     residual=dirichlet_residual, 
                                     residual_name="DirichletConstraint", 
                                     weight=50.0)
dirichlet_constraint = dirichlet_pipeline.constraint

Lastly the outflow condition, which we already finished for you:

In [10]:
def outflow_residual(u: U, p: P, x: X):
    normals = zeros_like_node(x)
    normals[:, :1] = 1.0
    sym_grad = mu * u.sym_grad(x)
    res = - sym_grad[:, :, 0] + p*normals
    return res

outflow_pipeline = qewton.PINNPipeline(outflow_boundary_sampler, [model],
                                     residual=outflow_residual,
                                     residual_name="OutflowConstraint")
outflow_constraint = outflow_pipeline.constraint

We can now start the training:

In [11]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=5000,
)

lbfgs_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.LBFGS(),
    lr=0.1,
    max_iterations=400,
    optimizer_args={"max_eval": 5},
)

# LBFGS needs to have the same data for all iterations, 
# so we cache the sample points after the Adam phase.
fix_sampler_callback = qewton.optim.CacheDataCallback(
    data_nodes=[inner_sampler, dirichlet_boundary_sampler, outflow_boundary_sampler],
    phase_to_start_cache=lbfgs_phase
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase, lbfgs_phase],
    graphs=[mass_pipeline, momentum_pipeline, 
            outflow_pipeline, dirichlet_pipeline],
    callbacks=[fix_sampler_callback],
    training_objectives=[mass_constraint, momentum_constraint, 
                         outflow_constraint, dirichlet_constraint],
    device=qewton.cuda(1),
)

trainer.run()

/home/tomfre/miniconda3/envs/PIONEER/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Optimization Phase 2: 100%|██████████| 1000/1000 [13:25<00:00,  1.24it/s, loss=0.00433]


We could train even more to further improve the results. But let us check the solution now:

In [13]:
plot = mass_pipeline.visualize(model.output_ports[0])
qewton.viz.Figure(plot).show()

NotImplementedError: 

In [38]:
reference_data = qewton.DEFAULT_DL_BACKEND.load_data(path + "solution_data.pt")

data_config = qewton.DataConfiguration(
        qewton.GeometryAxes(shape=(3,)), 
        qewton.GeometryAxes(shape=(33, 33)),
        qewton.FeatureAxes(variable=A*X*U)
)

plot = mass_pipeline.visualize(model.output_ports[0], 
                               data=reference_data, 
                               reference_config=data_config)
qewton.viz.Figure(plot).show()

NotImplementedError: 